In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# Load dataset
df = pd.read_csv("../data/raw/heart.csv")

# Show basic info
print(df.shape)
df.head()




(9651, 27)


,Age,Cholesterol,Heart rate,Diabetes,Family History,Smoking,Obesity,Alcohol Consumption,Exercise Hours Per Week,Diet,...,Physical Activity Days Per Week,Sleep Hours Per Day,Heart Attack Risk (Binary),Blood sugar,CK-MB,Troponin,Heart Attack Risk (Text),Gender,Systolic blood pressure,Diastolic blood pressure
0,0.595506,0.314286,0.047663,0.0,0.0,1.0,0.0,0.0,0.208326,0,...,0.0,0.333333,0.0,0.227018,0.048229,0.036512,0,Male,0.600000,0.534884
1,0.595506,0.096429,0.047663,1.0,1.0,1.0,1.0,1.0,0.752420,1,...,2.0,0.666667,0.0,0.227018,0.048229,0.036512,0,Male,0.574194,0.569767
2,0.595506,0.189286,0.047663,0.0,0.0,1.0,0.0,1.0,0.200998,2,...,4.0,1.000000,0.0,0.227018,0.048229,0.036512,0,Male,0.187097,0.674419
3,0.078652,0.960714,0.071494,1.0,1.0,1.0,1.0,1.0,0.090557,2,...,1.0,0.500000,0.0,0.227018,0.048229,0.036512,0,Male,0.645161,0.593023
4,0.078652,0.792857,0.071494,1.0,0.0,1.0,1.0,0.0,0.601030,2,...,1.0,0.166667,0.0,0.227018,0.048229,0.036512,0,Male,0.251613,0.383721


In [18]:
# Remove extra spaces in column names
df.columns = df.columns.str.strip()

# Rename target column for easier handling
if 'Heart Attack Risk (Binary)' in df.columns:
    df.rename(columns={'Heart Attack Risk (Binary)': 'Heart_Attack_Risk'}, inplace=True)

# Drop text column if exists
if 'Heart Attack Risk (Text)' in df.columns:
    df.drop(columns=['Heart Attack Risk (Text)'], inplace=True)

# Drop rows with missing target
df = df.dropna(subset=['Heart_Attack_Risk'])

# Fill missing numeric values with median
for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].median())

# Fill missing categorical values with mode
for col in df.select_dtypes(exclude=np.number).columns:
    df[col] = df[col].fillna(df[col].mode()[0])

print("✅ Data cleaned successfully")
print(df.shape)



✅ Data cleaned successfully
(9651, 26)


In [19]:
# Identify categorical columns
cat_cols = df.select_dtypes(exclude=np.number).columns.tolist()

# One-hot encode categorical features (safe encoding)
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print("✅ Encoding done. Final columns:", len(df.columns))



✅ Encoding done. Final columns: 28


In [20]:
#Clean Column Names

In [21]:
target = 'Heart_Attack_Risk'

X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)





Train size: (7720, 27)
Test size: (1931, 27)


In [22]:
#Handle Missing & Invalid Values

In [23]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

print("✅ Model training complete")



✅ Model training complete


In [24]:
#Encode Categorical Variables

In [25]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

    



Accuracy: 0.6758156395649922

Classification Report:
               precision    recall  f1-score   support

         0.0       0.68      0.97      0.80      1287
         1.0       0.60      0.08      0.15       644

    accuracy                           0.68      1931
   macro avg       0.64      0.53      0.47      1931
weighted avg       0.65      0.68      0.58      1931



In [26]:
#Scale Numeric Features

In [28]:
import os

# Create the models folder if it doesn't exist
os.makedirs("../data/models", exist_ok=True)

# Now save the model
joblib.dump(model, "../data/models/heart_attack_risk_model.pkl")
print("✅ Model saved successfully!")



✅ Model saved successfully!


In [29]:
import numpy as np

# Example new input (you can replace with real data)
new_data = {
    "Age": 45,
    "Cholesterol": 220,
    "Heart rate": 85,
    "Diabetes": 1,
    "Family History": 1,
    "Smoking": 0,
    "Obesity": 1,
    "Alcohol Consumption": 2,
    "Exercise Hours Per Week": 4,
    "Previous Heart Problems": 0,
    "Medication Use": 1,
    "Stress Level": 6,
    "Sedentary Hours Per Day": 8,
    "Income": 40000,
    "BMI": 29.5,
    "Triglycerides": 150,
    "Physical Activity Days Per Week": 3,
    "Sleep Hours Per Day": 6,
    "Blood sugar": 110,
    "CK-MB": 20,
    "Troponin": 0.04,
    "Gender": "Male",
    "Systolic blood pressure": 130,
    "Diastolic blood pressure": 85
}

# Convert to DataFrame
new_df = pd.DataFrame([new_data])

# Apply same preprocessing (encoding)
for col in new_df.select_dtypes(exclude=np.number).columns:
    new_df[col] = new_df[col].astype("category")

# Align columns with training data
new_df = pd.get_dummies(new_df).reindex(columns=X_train.columns, fill_value=0)

# Predict risk
pred = model.predict(new_df)[0]
print("Predicted Heart Attack Risk (1 = High Risk, 0 = Low Risk):", pred)


Predicted Heart Attack Risk (1 = High Risk, 0 = Low Risk): 1.0
